In [89]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [91]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [92]:
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [94]:
df['prompt'].str.len()

0       142
1        43
2       182
3       129
4       110
       ... 
1995     72
1996    104
1997     93
1998    266
1999    140
Name: prompt, Length: 2000, dtype: int64

In [95]:
df['prompt'].str.len().sum()/2000

np.float64(117.6695)

In [96]:
df['prompt'].str.count(':').value_counts().sort_index()

prompt
0     480
1    1511
2       9
Name: count, dtype: int64

In [97]:
df.loc[df['prompt'].str.count(':') == 2, 'prompt']

331     Identify the correct statement: What is the pu...
450     Identify the correct statement: What is the pu...
845     Determine the correct option: What is the purp...
1593    Determine the correct option: What is the purp...
1684    Pick the best possible answer: What is the pur...
1796    Select the most accurate option: What is the p...
1835    Select the most accurate option: What is the p...
1854    Pick the best possible answer: What is the pur...
1913    Pick the best possible answer: What is the pur...
Name: prompt, dtype: object

In [98]:
for i, prompt in enumerate(df.loc[df['prompt'].str.count(':') == 2, 'prompt'], 1):
    print(i, prompt, '\n')

1 Identify the correct statement: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? carefully. 

2 Identify the correct statement: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? among the listed options. 

3 Determine the correct option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

4 Determine the correct option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? based on the given context. 

5 Pick the best possible answer: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

6 Select the most accurate option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

7 Select the most accurate option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? 

8 Pick the best possible answer: What is the purpose of 

In [99]:
df['prompt'].str.count(r'\?').value_counts().sort_index()

prompt
1    1718
2     282
Name: count, dtype: int64

In [100]:
df.loc[df['prompt'].str.count(r'\?') == 2, 'prompt']

6       Which of the following is correct? What is a "...
9       Which of the following is correct? What is the...
19      Which of the following is correct? Who propose...
21      Which of the following is correct? Who shared ...
23      Which of the following is correct? Which of th...
                              ...                        
1948    Which of the following is correct? What is res...
1971    Which of the following is correct? What is the...
1975    Which of the following is correct? What is the...
1978    Which of the following is correct? What is the...
1991    Which of the following is correct? What is a t...
Name: prompt, Length: 282, dtype: object

In [101]:
for i, prompt in enumerate(df.loc[df['prompt'].str.count(r'\?') == 2, 'prompt'].head(10), 1):
    print(i, prompt, '\n')

1 Which of the following is correct? What is a "coffee ring" in physics? among the listed options. 

2 Which of the following is correct? What is the relationship between the Wigner function and the density matrix operator? from the following choices. 

3 Which of the following is correct? Who proposed the concept of "maximal acceleration"? carefully. 

4 Which of the following is correct? Who shared the other half of the Nobel Prize with Yoichiro Nambu for discovering the origin of the explicit breaking of CP symmetry in the weak interactions? among the listed options. 

5 Which of the following is correct? Which of the following statements accurately describes the blocking temperature of an antiferromagnetic layer in a spin valve? carefully. 

6 Which of the following is correct? What is the Ramsauer-Townsend effect? carefully. 

7 Which of the following is correct? What is the reason for the stochastic nature of all observed resistance-switching processes? from the following choices

In [102]:
df.loc[df['prompt'].str.count(r'\?') == 1, 'prompt']

0       Pick the best possible answer: What is Martin ...
1             What is accelerator-based light-ion fusion?
2       Determine the correct option: What is the term...
3       Select the most accurate option: What is Marti...
4       Identify the correct statement: What is the co...
                              ...                        
1995    What is the piezoelectric strain coefficient f...
1996    Identify the correct statement: What is the sy...
1997    Determine the correct option: What does Earnsh...
1998    Identify the correct statement: What is the re...
1999    Pick the best possible answer: Which of the fo...
Name: prompt, Length: 1718, dtype: object

In [103]:
# vec = CountVectorizer(ngram_range=(3, 7), analyzer="word")
# X = vec.fit_transform(df["prompt"].str.lower())

# freq = np.asarray(X.sum(axis=0)).flatten()
# vocab = vec.get_feature_names_out()

In [104]:
# ngram_freq = pd.Series(freq, index=vocab).sort_values(ascending=False)
# print(ngram_freq[ngram_freq > 1].head(20))

In [105]:
prefixes, questions, suffixes = [], [], []

for text in df['prompt']:
    text = text.strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    prefixes.append(prefix or None)
    questions.append(text)
    suffixes.append(suffix or None)

df['prefix'] = prefixes
df['question'] = questions
df['suffix'] = suffixes

print(df['prefix'].value_counts(dropna=False))
print(df['suffix'].value_counts(dropna=False))

prefix
Pick the best possible answer         316
Select the most accurate option       309
Identify the correct statement        302
Determine the correct option          299
Choose the correct answer             292
Which of the following is correct?    282
None                                  200
Name: count, dtype: int64
suffix
None                           538
among the listed options.      388
based on the given context.    376
from the following choices.    369
carefully.                     329
Name: count, dtype: int64


In [106]:
df['prompt'].duplicated().sum()

np.int64(242)

In [107]:
df['question'].duplicated().sum()

np.int64(1748)

In [108]:
df.groupby('question')['answer'].nunique().max()

1

In [109]:
df['correct_text'] = df.apply(lambda r: r[r['answer']], axis=1)

df.groupby('question')['correct_text'].nunique().max()

9

In [110]:
df.groupby('question')['correct_text'].nunique().sort_values(ascending=False).head(20)

question
What is the Erlangen program?                                                                                                                                                      9
What is the Maxwell's Demon thought experiment?                                                                                                                                    8
What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"?                                                                                             6
What is accelerator-based light-ion fusion?                                                                                                                                        4
What is radiometric dating?                                                                                                                                                        4
What is the application of Memristor?                                                 